## Cube preparations for SCS3: Model-Data Fusion for Understanding Carbon State-Flux Relationships Across Space in the EO-LINCS project

This notebook prepares and inspects **Sindbad-ready forcing / observation inputs** from site-level Zarr cubes (ERA5-Land, optionally Sentinel-2 and biomass products), and performs quick sanity checks before saving the merged dataset.

## What this notebook does
- Opens source Zarr datasets (ERA5-Land and optional auxiliary cubes)
- Standardizes dimensions and coordinates for a **single-site** workflow
- Builds a Sindbad-style dataset layout (time-varying fields, static fields, soil-profile fields)
- Merges auxiliary variables (e.g., Sentinel-2 / biomass) into the forcing dataset
- Rechunks and writes the final dataset to a Zarr store
- Runs basic inspection checks (variable lists, non-NaN values, quick dataframe views)

## Expected usage
This notebook is intended for **interactive preprocessing/debugging** of a site (e.g., `AU-Dry`) before running Sindbad experiments on HPC.

## Important assumptions
- Input Zarr stores are already generated and accessible on the cluster filesystem.
- Variable names and dimensions follow the expected conventions in the helper functions below.
- Output paths are writable.


## 1. Imports

Load the core Python libraries used in this preprocessing workflow:
- **pandas** for tabular inspection
- **xarray** for labeled multidimensional arrays and Zarr I/O
- **numpy** for numerical operations and masking


In [1]:
import pandas as pd
import xarray as xr
import numpy as np

## 2. Forcing variable summary and source metadata

This cell documents the expected forcing variables, dimensions, and source paths used in the workflow.  
Treat it as a **reference cell** describing the schema and conventions (units, dimensions, and mapping to Sindbad inputs).


In [ ]:
# -----------------------------------------------------------------------------
# Forcing variables summary (from forcing JSON)
#
# Dataset / dimensions
# - data_path (default): ../data/FLUXNET_v2023_12_1D.zarr
# - time dimension: "Ti"
# - space dimension: ["site"]
# - default space_time_type: spatiotemporal
#
# Essential forcing variables for model runs (f_* -> source_variable)
#
# 1) Atmospheric / meteorological forcing (time-varying)
# - f_ambient_CO2  -> atmCO2_SCRIPPS_global      [ppm]
#   Ambient CO2 concentration
# - f_airT         -> TA_ERAIv2_gfld             [°C]
#   Air temperature
# - f_airT_day     -> TA_DayTime_ERAIv2_gfld     [°C]
#   Daytime air temperature
# - f_VPD          -> VPD_ERAIv2_gfld            [Pa -> kPa, x0.001]
#   Vapor pressure deficit
# - f_VPD_day      -> VPD_DayTime_ERAIv2_gfld    [Pa -> kPa, x0.001]
#   Daytime vapor pressure deficit
# - f_rain         -> P_ERAIv2_gfld              [mm d-1]
#   Rain / precipitation
#
# 2) Radiation forcing (time-varying)
# - f_rg           -> SW_IN_ERAIv2_gfld          [MJ m-2 d-1] [SSR]
#   Global shortwave radiation
# - f_PAR          -> SW_IN_ERAIv2_gfld          [MJ m-2 d-1, x0.5]
#   Photosynthetically active radiation (derived as Rg * 0.5)
# - f_rg_pot       -> SW_IN_POT_ONEFlux          [MJ m-2 d-1]
#   Potential global radiation
# - f_rn           -> NETRAD_ERAIv2_gfld         [MJ m-2 d-1] [SSR]
#   Net radiation
#
# 3) Static / site / soil / vegetation descriptors (often spatiovertical)
# - f_pft              -> f_pft                  [categorical index 1..17]
#   Plant functional type index
# - f_frac_vegetation  -> veg_frac               [0-1]
#   Vegetation fraction
# - f_tree_frac        -> tree_frac              [0-1]
#   Tree fraction
# - f_dist_intensity   -> dist_frac_sb2018       [categorical/flag]
#   Disturbance flag / intensity
# - f_burnt_area       -> fire_frac              [0-1]
#   Burnt area fraction
#
# Soil texture / properties (SoilGrids; percent values converted to fraction with x0.01 where noted)
# - f_clay         -> CLYPPT_SoilGrids           [% -> -, x0.01]
# - f_sand         -> SNDPPT_SoilGrids           [% -> -, x0.01]
# - f_silt         -> SLTPPT_SoilGrids           [% -> -, x0.01]
# - f_orgm         -> OCSTHA_SoilGrids           [configured x0.0 in JSON; check conversion!]
#   NOTE: source_to_sindbad_unit=0.0 looks suspicious and may zero-out this variable.
#
# Practical "minimum core" forcing set (most runs)
# - CO2:        f_ambient_CO2
# - Met:        f_airT (and/or f_airT_day), f_VPD (and/or f_VPD_day), f_rain
# - Radiation:  f_rg and/or f_PAR, optionally f_rg_pot, f_rn
# - Land cover: f_pft, f_frac_vegetation, f_tree_frac
# - Soil:       f_clay, f_sand, f_silt (optionally f_orgm after checking conversion)
# - Disturbance: f_dist_intensity, f_burnt_area (if disturbance/fire processes are enabled)
#
# Important notes
# - f_PAR and f_rg use the same source variable (SW_IN_ERAIv2_gfld); f_PAR applies a 0.5 factor.
# - Several static variables are marked "spatiovertical" (site/depth-related structure).
# - Bounds are provided in JSON and can be used for sanity checks / masking.
# -----------------------------------------------------------------------------

## 3. Environment / path sanity check

This cell is used to confirm the current working directory and/or notebook execution context before reading files from absolute or relative paths.


In [2]:
# load file
!pwd

/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation


## 4. Open and inspect ERA5-Land Zarr cube

This section loads the site-level ERA5-Land Zarr dataset and performs initial inspection:
- prints dataset structure and variables
- reads core meteorological variables (e.g., `t2m`, `d2m`)
- optionally squeezes singleton dimensions if present (e.g., `expver`, `number`)

Use this step to verify dimensions, coordinate names, and variable availability before conversion/merge.


In [3]:
# Path to zarr store
zarr_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land.zarr"

# Open dataset
ds = xr.open_zarr(zarr_path, consolidated=False)  # try consolidated=True if needed

print(ds)
print("Variables:", list(ds.data_vars))

# ERA5 variables (usually in Kelvin)
t2m = ds["t2m"]
d2m = ds["d2m"]

# Optional: remove singleton dimensions like expver / number if present
# (keep this generic so it doesn't break if dims are absent)
for dim in ["expver", "number"]:
    if dim in t2m.dims and t2m.sizes[dim] == 1:
        t2m = t2m.isel({dim: 0}, drop=True)
    if dim in d2m.dims and d2m.sizes[dim] == 1:
        d2m = d2m.isel({dim: 0}, drop=True)

# Convert K -> degC
t2m_c = t2m - 273.15
d2m_c = d2m - 273.15

# Magnus formula for saturation vapor pressure (Pa)
# es(T) and ea(Td)
es = 611.2 * np.exp((17.67 * t2m_c) / (t2m_c + 243.5))
ea = 611.2 * np.exp((17.67 * d2m_c) / (d2m_c + 243.5))

# Vapor Pressure Deficit
vpd_pa = es - ea
vpd_pa = xr.where(vpd_pa < 0, 0, vpd_pa)  # numerical safety

# Also in kPa (common in ecohydrology / land surface models)
vpd_kpa = vpd_pa / 1000.0

# Add to dataset
ds["VPD"] = vpd_pa
ds["VPD"].attrs = {
    "long_name": "Vapor Pressure Deficit",
    "units": "Pa",
    "description": "Computed from ERA5-Land t2m and d2m using Magnus equation"
}

ds["VPD_kPa"] = vpd_kpa
ds["VPD_kPa"].attrs = {
    "long_name": "Vapor Pressure Deficit",
    "units": "kPa",
    "description": "Computed from ERA5-Land t2m and d2m using Magnus equation"
}

print(ds["VPD"])
print(ds["VPD_kPa"])

# Example quick summary
print("VPD_kPa min/max:", float(ds["VPD_kPa"].min().compute()), float(ds["VPD_kPa"].max().compute()))

<xarray.Dataset> Size: 387kB
Dimensions:  (time: 8784)
Coordinates:
  * time     (time) datetime64[ns] 70kB 2020-01-01 ... 2020-12-31T23:00:00
    expver   (time) <U4 141kB dask.array<chunksize=(552,), meta=np.ndarray>
    lat      float64 8B ...
    lon      float64 8B ...
    number   int64 8B ...
Data variables:
    d2m      (time) float32 35kB dask.array<chunksize=(552,), meta=np.ndarray>
    ssr      (time) float32 35kB dask.array<chunksize=(552,), meta=np.ndarray>
    ssrd     (time) float32 35kB dask.array<chunksize=(552,), meta=np.ndarray>
    t2m      (time) float32 35kB dask.array<chunksize=(552,), meta=np.ndarray>
    tp       (time) float32 35kB dask.array<chunksize=(552,), meta=np.ndarray>
Attributes:
    Conventions:             CF-1.7
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    history:                 2026-02-22T10:01 GRIB to CDM+CF via cfgrib-0.9.1...
    institutio

/User/homes/xshan/miniforge3/envs/xcube-multistore/lib/python3.13/site-packages/dask/array/chunk.py:279: RuntimeWarning: invalid value encountered in cast
  return x.astype(astype_dtype, **kwargs)


## 5. Build a Sindbad-style forcing dataset (ERA5-Land only)

This section defines helper functions and constructs a **Sindbad-compatible single-site forcing dataset** from ERA5-Land.

### Target shape conventions (as noted in code)
- **Time-varying meteorology/radiation/CO2**: `[time, lat, lon]`
- **Static / categorical surface variables**: `[time, lat, lon]` (broadcast in time)
- **Soil profile variables**: `[depth, lat, lon]`

This cell is the core preprocessing logic for transforming raw inputs into a modeling-ready format.


In [5]:
import xarray as xr
import numpy as np


# =============================================================================
# Prepare SINDBAD-style forcing dataset from ERA5-Land Zarr (single-site cube)
#
# Final shape convention:
#   - time-varying meteorology/radiation/CO2: [time, lat, lon]  (lat=1, lon=1)
#   - static/categorical surface vars:        [time, lat, lon]  (broadcast in time)
#   - soil profile vars:                      [depth, lat, lon] (depth=7, lat=1, lon=1)
#
# AU-Dry PFT = SAV = 9 (based on SindbadML.PFTlabels, 1-based indexing)
# =============================================================================


# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------
zarr_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land.zarr"

# If kowning the site coordinate, set it here and use nearest grid-cell selection.
# Otherwise, set to None and the script will use the first grid cell.
site_lat = None   # e.g., -23.5
site_lon = None   # e.g., 133.9

# SoilGrids standard 7 depths (cm), from the screenshot metadata
SOIL_DEPTH_CM = np.array([0.0, 5.0, 15.0, 30.0, 60.0, 100.0, 200.0], dtype=np.float32)

# AU-Dry = SAV
PFT_AU_DRY = 9

# Placeholder constants (JSON/SINDBAD compatible units)
# NOTE: these are fallback values only, for variables not present in the ERA5 zarr.
CONST = {
    # Spatiotemporal forcing [time,lat,lon]
    "f_ambient_CO2": 420.0,   # ppm

    # Static/categorical surface vars, but will be broadcast to [time,lat,lon]
    "f_frac_vegetation": 0.8, # unitless fraction [0-1]
    "f_tree_frac": 0.25,      # unitless fraction [0-1]
    "f_pft": PFT_AU_DRY,      # SAV
    "f_dist_intensity": 0,    # categorical/flag
    "f_burnt_area": 0.0,      # unitless fraction [0-1]

    # Soil profile vars [depth,lat,lon], in Sindbad-compatible units (fractions)
    "f_clay": 0.20,           # fraction
    "f_sand": 0.70,           # fraction
    "f_silt": 0.10,           # fraction
    "f_orgm": 0.00,           # placeholder
}


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def squeeze_singleton_dims(da, dims=("expver", "number")):
    """Drop singleton dims often found in ERA5 stores."""
    for d in dims:
        if d in da.dims and da.sizes[d] == 1:
            da = da.isel({d: 0}, drop=True)
    return da


def add_if_missing(ds, name, da, attrs=None):
    """Add variable only if missing."""
    if name not in ds:
        ds[name] = da
        if attrs:
            ds[name].attrs.update(attrs)
        print(f"Added: {name:18s} dims={ds[name].dims} shape={ds[name].shape}")
    else:
        print(f"Exists, skip: {name}")
    return ds


def ensure_time_dim_name(da, target="time"):
    """Rename a time-like dim to 'time' if needed."""
    if target in da.dims:
        return da
    time_like = [d for d in da.dims if d.lower() == "time"]
    if len(time_like) == 1:
        return da.rename({time_like[0]: target})
    return da


def ensure_time_lat_lon_shape(da, lat_val=np.nan, lon_val=np.nan):
    """
    Force a DataArray to have dims [time, lat, lon] with singleton lat/lon if needed.
    If lat/lon exist and size > 1, caller should subset before calling.
    """
    da = squeeze_singleton_dims(da)
    da = ensure_time_dim_name(da, target="time")

    if "time" not in da.dims:
        raise ValueError(f"Expected time dimension, got dims={da.dims}")

    if "lat" not in da.dims:
        da = da.expand_dims(lat=[lat_val])
    if "lon" not in da.dims:
        da = da.expand_dims(lon=[lon_val])

    # Reorder exactly
    da = da.transpose("time", "lat", "lon")

    # Overwrite coords with explicit singleton coords if needed
    if da.sizes["lat"] == 1:
        da = da.assign_coords(lat=[lat_val])
    if da.sizes["lon"] == 1:
        da = da.assign_coords(lon=[lon_val])

    return da


def const_3d_time_lat_lon(time_coord, lat_val, lon_val, value, dtype=np.float32):
    """Create [time,lat,lon] constant field."""
    data = np.full((time_coord.size, 1, 1), value, dtype=dtype)
    return xr.DataArray(
        data,
        dims=("time", "lat", "lon"),
        coords={"time": time_coord, "lat": [lat_val], "lon": [lon_val]},
    )


def const_3d_depth_lat_lon(depth_vals, lat_val, lon_val, value, dtype=np.float32):
    """Create [depth,lat,lon] constant field."""
    data = np.full((len(depth_vals), 1, 1), value, dtype=dtype)
    return xr.DataArray(
        data,
        dims=("depth", "lat", "lon"),
        coords={"depth": depth_vals, "lat": [lat_val], "lon": [lon_val]},
    )


def ensure_time_lat_lon_broadcast(da, time_coord, lat_val=np.nan, lon_val=np.nan):
    """
    Ensure DataArray has dims [time, lat, lon].
    - If [lat, lon], broadcast to time.
    - If [time, lat, lon], reorder and return.
    - If scalar, expand to [time,lat,lon].
    """
    da = squeeze_singleton_dims(da)
    da = ensure_time_dim_name(da, target="time")

    if da.ndim == 0:
        # scalar
        value = da.values.item()
        return const_3d_time_lat_lon(time_coord, lat_val, lon_val, value, dtype=da.dtype)

    dims_set = set(da.dims)

    if dims_set == {"lat", "lon"}:
        da = da.assign_coords(lat=[lat_val] if da.sizes.get("lat", 0) == 1 else da["lat"],
                              lon=[lon_val] if da.sizes.get("lon", 0) == 1 else da["lon"])
        da = da.expand_dims(time=time_coord).transpose("time", "lat", "lon")
        if da.sizes["lat"] == 1:
            da = da.assign_coords(lat=[lat_val])
        if da.sizes["lon"] == 1:
            da = da.assign_coords(lon=[lon_val])
        return da

    if dims_set == {"time", "lat", "lon"}:
        da = da.transpose("time", "lat", "lon")
        if da.sizes["lat"] == 1:
            da = da.assign_coords(lat=[lat_val])
        if da.sizes["lon"] == 1:
            da = da.assign_coords(lon=[lon_val])
        return da

    raise ValueError(f"Unexpected dims for time broadcast: {da.dims}")


# -----------------------------------------------------------------------------
# Open dataset
# -----------------------------------------------------------------------------
ds = xr.open_zarr(zarr_path, consolidated=False)  # set True if zarr is consolidated

# Squeeze singleton dims for all variables
for v in list(ds.data_vars):
    ds[v] = squeeze_singleton_dims(ds[v])

# Ensure time coordinate exists
if "time" not in ds.coords:
    # try to find time-like coord and rename
    time_like_coords = [c for c in ds.coords if c.lower() == "time"]
    if len(time_like_coords) == 1:
        ds = ds.rename({time_like_coords[0]: "time"})
    else:
        raise ValueError("No 'time' coordinate found in dataset.")

time_coord = ds["time"]

# -----------------------------------------------------------------------------
# Select a single grid cell and force time-varying variables to [time,lat,lon]
# -----------------------------------------------------------------------------
# Determine target lat/lon
if ("lat" in ds.coords) and ("lon" in ds.coords):
    if (site_lat is not None) and (site_lon is not None):
        # Nearest site selection
        ds = ds.sel(lat=site_lat, lon=site_lon, method="nearest")
        lat_val = float(ds["lat"].values) if np.ndim(ds["lat"].values) == 0 else float(ds["lat"].values[0])
        lon_val = float(ds["lon"].values) if np.ndim(ds["lon"].values) == 0 else float(ds["lon"].values[0])
    else:
        # First grid cell
        lat_idx = 0
        lon_idx = 0
        if ds["lat"].size > 1 or ds["lon"].size > 1:
            ds = ds.isel(lat=lat_idx, lon=lon_idx)
        lat_val = float(ds["lat"].values) if np.ndim(ds["lat"].values) == 0 else float(ds["lat"].values[0])
        lon_val = float(ds["lon"].values) if np.ndim(ds["lon"].values) == 0 else float(ds["lon"].values[0])
else:
    # No lat/lon coords in source; still build singleton coords
    lat_val = np.nan
    lon_val = np.nan

# Re-squeeze after selection
for v in list(ds.data_vars):
    ds[v] = squeeze_singleton_dims(ds[v])

# Force all time-varying source vars to [time,lat,lon]
for v in list(ds.data_vars):
    da = ds[v]
    da = ensure_time_dim_name(da, "time")
    if "time" in da.dims:
        ds[v] = ensure_time_lat_lon_shape(da, lat_val=lat_val, lon_val=lon_val)

# -----------------------------------------------------------------------------
# Compute derived forcing variables from ERA5-Land
# ERA5 variable assumptions:
#   t2m  [K]
#   d2m  [K]
#   tp   [m water equivalent over timestep]
#   ssrd [J m-2 over timestep]      downward shortwave radiation
#   OPTIONAL for net radiation:
#     ssr [J m-2 over timestep]     net shortwave radiation (surface solar radiation net)
#     str [J m-2 over timestep]     net longwave radiation (surface thermal radiation net)
# -----------------------------------------------------------------------------
t2m = ds["t2m"] if "t2m" in ds else None
d2m = ds["d2m"] if "d2m" in ds else None
tp = ds["tp"] if "tp" in ds else None
ssrd = ds["ssrd"] if "ssrd" in ds else None
ssr = ds["ssr"] if "ssr" in ds else None
str_ = ds["str"] if "str" in ds else None  # avoid shadowing Python built-in str

# f_airT [°C]
if t2m is not None:
    f_airT = (t2m - 273.15).astype(np.float32)
    ds = add_if_missing(ds, "f_airT", f_airT, attrs={
        "standard_name": "Tair",
        "units": "°C",
        "source_variable": "t2m",
        "note": "Converted from K to °C",
    })

# f_airT_day [°C] (fallback = f_airT)
if "f_airT" in ds:
    ds = add_if_missing(ds, "f_airT_day", ds["f_airT"].astype(np.float32), attrs={
        "standard_name": "TairDay",
        "units": "°C",
        "source_variable": "t2m",
        "note": "Fallback = f_airT (no separate daytime T available)",
    })

# f_VPD [kPa] and f_VPD_day [kPa] from t2m,d2m using Magnus equation
#   es(T) = 611.2 * exp(17.67*T / (T + 243.5))         [Pa]
#   ea(Td)= 611.2 * exp(17.67*Td / (Td + 243.5))       [Pa]
#   VPD   = max(0, es(T) - ea(Td)) / 1000              [kPa]
if (t2m is not None) and (d2m is not None):
    t_c = t2m - 273.15
    td_c = d2m - 273.15

    es = 611.2 * np.exp((17.67 * t_c) / (t_c + 243.5))    # Pa
    ea = 611.2 * np.exp((17.67 * td_c) / (td_c + 243.5))  # Pa

    vpd_pa = xr.where((es - ea) < 0, 0, es - ea)
    vpd_kpa = (vpd_pa / 1000.0).astype(np.float32)

    ds = add_if_missing(ds, "f_VPD", vpd_kpa, attrs={
        "standard_name": "Vapor pressure deficit",
        "units": "kPa",
        "source_variable": "t2m,d2m",
        "formula": "VPD = es(Tair)-ea(Tdew) using Magnus equation",
    })

    ds = add_if_missing(ds, "f_VPD_day", vpd_kpa, attrs={
        "standard_name": "Vapor pressure deficit Day",
        "units": "kPa",
        "source_variable": "t2m,d2m",
        "note": "Fallback = f_VPD (no separate daytime VPD inputs available)",
    })

# f_rg [MJ m-2 timestep-1] and f_PAR [MJ m-2 timestep-1]
# ERA5 ssrd is typically accumulated energy [J m-2] over the native timestep
if ssrd is not None:
    f_rg = (ssrd / 1e6).astype(np.float32)  # J -> MJ
    ds = add_if_missing(ds, "f_rg", f_rg, attrs={
        "standard_name": "Global Radiation",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Converted from J m-2 to MJ m-2; still per timestep accumulation",
    })

    f_PAR = (0.5 * f_rg).astype(np.float32)
    ds = add_if_missing(ds, "f_PAR", f_PAR, attrs={
        "standard_name": "Photosynthetically active radiation (=Rg*.5)",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Approximated as 0.5 * f_rg",
    })

# f_rn [MJ m-2 timestep-1] (Net radiation)
# Preferred ERA5 route:
#   f_rn = (ssr + str) / 1e6
# where:
#   ssr = surface net solar radiation (positive downward)
#   str = surface net thermal radiation (often negative at surface)
#
# If ssr/str are unavailable but ssrd exists, fallback to f_rg as approximation.
if (ssr is not None) and (str_ is not None):
    f_rn = ((ssr + str_) / 1e6).astype(np.float32)  # J -> MJ
    ds = add_if_missing(ds, "f_rn", f_rn, attrs={
        "standard_name": "Net radiation",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssr,str",
        "formula": "Rn = (SSR + STR) / 1e6",
        "note": "ERA5 accumulated net shortwave + net longwave over timestep",
    })
elif "f_rg" in ds:
    # Approximate fallback if only shortwave downward is present
    ds = add_if_missing(ds, "f_rn", ds["f_rg"].astype(np.float32), attrs={
        "standard_name": "Net radiation (approximate)",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Fallback approximation f_rn = f_rg because ssr/str not available; ignores net longwave and reflected shortwave",
    })

# f_rain [mm timestep-1]
# ERA5 tp is typically total precipitation [m] over timestep
if tp is not None:
    f_rain = (tp * 1000.0).astype(np.float32)  # m -> mm
    ds = add_if_missing(ds, "f_rain", f_rain, attrs={
        "standard_name": "Rain",
        "units": "mm timestep-1",
        "source_variable": "tp",
        "note": "Converted from m to mm; accumulation interval depends on dataset timestep",
    })


# -----------------------------------------------------------------------------
# Add missing constants in the required shapes
#   - spatiotemporal: [time,lat,lon]
#   - static/categorical surface (broadcasted): [time,lat,lon]
#   - soil profile: [depth,lat,lon]
# -----------------------------------------------------------------------------
spatiotemporal_constants = {
    "f_ambient_CO2": (
        CONST["f_ambient_CO2"], np.float32,
        {
            "standard_name": "ambient_CO2",
            "units": "ppm",
            "source_variable": "constant_fill",
            "note": "Placeholder constant compatible with JSON",
        }
    ),
}

# Per requirement, these are ALSO [time,lat,lon] (broadcasted in time)
surface_constants_time_broadcast = {
    "f_frac_vegetation": (
        CONST["f_frac_vegetation"], np.float32,
        {
            "standard_name": "vegetation fraction",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "Static surface variable broadcasted to [time,lat,lon]",
        }
    ),
    "f_tree_frac": (
        CONST["f_tree_frac"], np.float32,
        {
            "standard_name": "tree fraction",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "Static surface variable broadcasted to [time,lat,lon]",
        }
    ),
    "f_pft": (
        CONST["f_pft"], np.int16,
        {
            "standard_name": "pft index",
            "units": "-",
            "source_variable": "constant_fill",
            "pft_label": "SAV",
            "pft_index_mapping": "SindbadML.PFTlabels (1-based)",
            "note": "Static categorical variable broadcasted to [time,lat,lon]",
        }
    ),
    "f_dist_intensity": (
        CONST["f_dist_intensity"], np.int16,
        {
            "standard_name": "isDisturbed flag for disturbance",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "Static categorical variable broadcasted to [time,lat,lon]",
        }
    ),
    "f_burnt_area": (
        CONST["f_burnt_area"], np.float32,
        {
            "standard_name": "fire fraction area per area pixel",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "Static surface variable broadcasted to [time,lat,lon]",
        }
    ),
}

soil_profile_constants = {
    "f_clay": (
        CONST["f_clay"], np.float32,
        {
            "standard_name": "CLAY",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "7-depth soil profile placeholder in Sindbad-compatible fraction",
        }
    ),
    "f_sand": (
        CONST["f_sand"], np.float32,
        {
            "standard_name": "SAND",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "7-depth soil profile placeholder in Sindbad-compatible fraction",
        }
    ),
    "f_silt": (
        CONST["f_silt"], np.float32,
        {
            "standard_name": "SILT",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "7-depth soil profile placeholder in Sindbad-compatible fraction",
        }
    ),
    "f_orgm": (
        CONST["f_orgm"], np.float32,
        {
            "standard_name": "organic matter",
            "units": "-",
            "source_variable": "constant_fill",
            "note": "7-depth soil profile placeholder",
        }
    ),
}

# Add spatiotemporal constants [time,lat,lon]
for name, (val, dtype, attrs) in spatiotemporal_constants.items():
    if name not in ds:
        da = const_3d_time_lat_lon(time_coord, lat_val, lon_val, val, dtype=dtype)
        ds = add_if_missing(ds, name, da, attrs=attrs)

# Add static/categorical surface constants as [time,lat,lon]
for name, (val, dtype, attrs) in surface_constants_time_broadcast.items():
    if name not in ds:
        da = const_3d_time_lat_lon(time_coord, lat_val, lon_val, val, dtype=dtype)
        ds = add_if_missing(ds, name, da, attrs=attrs)

# Add soil profile constants as [depth,lat,lon]
for name, (val, dtype, attrs) in soil_profile_constants.items():
    if name not in ds:
        da = const_3d_depth_lat_lon(SOIL_DEPTH_CM, lat_val, lon_val, val, dtype=dtype)
        ds = add_if_missing(ds, name, da, attrs=attrs)


# -----------------------------------------------------------------------------
# If any of the static/categorical vars already exist but are [lat,lon], broadcast
# them to [time,lat,lon] to satisfy the requirement.
# -----------------------------------------------------------------------------
for v in ["f_pft", "f_tree_frac", "f_frac_vegetation", "f_dist_intensity", "f_burnt_area"]:
    if v in ds and ds[v].dims != ("time", "lat", "lon"):
        ds[v] = ensure_time_lat_lon_broadcast(ds[v], time_coord, lat_val=lat_val, lon_val=lon_val)
        print(f"Broadcasted existing {v} to [time,lat,lon]: shape={ds[v].shape}")


# # -----------------------------------------------------------------------------
# # Optional aliases (only if downstream code expects them)
# # -----------------------------------------------------------------------------
# # Alias f_CO2 -> f_ambient_CO2
# if ("f_ambient_CO2" in ds) and ("f_CO2" not in ds):
#     ds["f_CO2"] = ds["f_ambient_CO2"]
#     ds["f_CO2"].attrs.update({
#         "units": "ppm",
#         "note": "Alias of f_ambient_CO2",
#     })
#     print("Added alias: f_CO2 -> f_ambient_CO2")

# # Optional aliases to old source-like names
# if ("f_rg" in ds) and ("SW_IN_ERAIv2_gfld" not in ds):
#     ds["SW_IN_ERAIv2_gfld"] = ds["f_rg"]
#     ds["SW_IN_ERAIv2_gfld"].attrs.update({"note": "Alias of f_rg"})
#     print("Added alias: SW_IN_ERAIv2_gfld -> f_rg")

# if ("f_rain" in ds) and ("P_ERAIv2_gfld" not in ds):
#     ds["P_ERAIv2_gfld"] = ds["f_rain"]
#     ds["P_ERAIv2_gfld"].attrs.update({"note": "Alias of f_rain"})
#     print("Added alias: P_ERAIv2_gfld -> f_rain")


# -----------------------------------------------------------------------------
# Final checks
# -----------------------------------------------------------------------------
print("\n================ FINAL DATASET SUMMARY ================")
print(ds)

print("\n================ VARIABLE SHAPES ======================")
for v in ds.data_vars:
    print(f"{v:20s} dims={ds[v].dims} shape={ds[v].shape} dtype={ds[v].dtype} units={ds[v].attrs.get('units', 'NA')}")

print("\n=========== REQUIRED SHAPE CHECKS (examples) ==========")
for v in ["f_airT", "f_airT_day", "f_VPD", "f_VPD_day", "f_rain", "f_rg", "f_rn", "f_PAR", "f_ambient_CO2",
          "f_pft", "f_tree_frac", "f_frac_vegetation", "f_dist_intensity", "f_burnt_area"]:
    if v in ds:
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims}")

for v in ["f_clay", "f_sand", "f_silt", "f_orgm"]:
    if v in ds:
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims}, depth={ds[v]['depth'].values}")

Added: f_airT             dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_airT_day         dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_VPD              dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_VPD_day          dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_rg               dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_PAR              dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_rain             dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_ambient_CO2      dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_frac_vegetation  dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_tree_frac        dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_pft              dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_dist_intensity   dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_burnt_area       dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_clay             dims=('depth', 'lat', 'lo

## 6. Build a Sindbad-style forcing dataset with Sentinel-2 and biomass merges

This section extends the base ERA5-Land forcing workflow by merging additional sources (e.g., **Sentinel-2** and **CCI biomass**) into the output dataset.

### Notes
- Biomass may be merged as a **spatial mean** (depending on the helper logic in this cell).
- Prefixing/renaming is used to preserve provenance and avoid variable name collisions.
- Coordinate harmonization and single-pixel selection are critical here.


In [5]:
# including sentinel2 and biomass data
# ERA5-Land forcing + biomass (CCI) + Sentinel-2 merge
# Biomass is merged as SPATIAL MEAN over x,y for each time step.

import xarray as xr
import numpy as np


# =============================================================================
# Prepare SINDBAD-style forcing dataset from ERA5-Land Zarr (single-site cube)
# + merge biomass (CCI) and Sentinel-2 Zarrs
#
# Final shape convention:
#   - time-varying meteorology/radiation/CO2: [time, lat, lon]  (lat=1, lon=1)
#   - static/categorical surface vars:        [time, lat, lon]  (broadcast in time)
#   - soil profile vars:                      [depth, lat, lon] (depth=7, lat=1, lon=1)
#   - merged remote sensing vars:             [time, lat, lon]  (lat=1, lon=1)
#
# NOTE:
#   - Biomass zarr (CCI): spatial mean over x,y is computed for each time step
#   - Sentinel-2 zarr: by default still selects ONE pixel (first pixel unless site_x/site_y given)
#   - s2_NDVI / s2_NDVI_masked below are computed as SPATIAL MEAN over x,y per time step
# =============================================================================


# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------
era5_zarr_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land.zarr"
biomass_zarr_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_ccibiomass.zarr"
sen2_zarr_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_sen2.zarr"

# If the site coords are known, set them. Otherwise first ERA5 pixel is used.
site_lat = None
site_lon = None

# Optional projected x/y for Sentinel/biomass zarrs (if nearest pixel selection there).
# For biomass spatial-mean merge, these are NOT used.
sen2_site_x = None
sen2_site_y = None

# SoilGrids standard 7 depths (cm)
SOIL_DEPTH_CM = np.array([0.0, 5.0, 15.0, 30.0, 60.0, 100.0, 200.0], dtype=np.float32)

# AU-Dry = SAV (SindbadML.PFTlabels, 1-based indexing)
PFT_AU_DRY = 9

# Placeholder constants (fallback only)
CONST = {
    # [time,lat,lon]
    "f_ambient_CO2": 420.0,   # ppm

    # static/categorical surface vars, but broadcast to [time,lat,lon]
    "f_frac_vegetation": 0.8,
    "f_tree_frac": 0.25,
    "f_pft": PFT_AU_DRY,
    "f_dist_intensity": 0,
    "f_burnt_area": 0.0,

    # soil profile [depth,lat,lon] in Sindbad-compatible units (fraction)
    "f_clay": 0.30,
    "f_sand": 0.40,
    "f_silt": 0.30,
    "f_orgm": 0.00,
}


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def squeeze_singleton_dims(da, dims=("expver", "number")):
    for d in dims:
        if d in da.dims and da.sizes[d] == 1:
            da = da.isel({d: 0}, drop=True)
    return da


def add_if_missing(ds, name, da, attrs=None):
    if name not in ds:
        ds[name] = da
        if attrs:
            ds[name].attrs.update(attrs)
        print(f"Added: {name:20s} dims={ds[name].dims} shape={ds[name].shape}")
    else:
        print(f"Exists, skip: {name}")
    return ds


def ensure_time_dim_name(da, target="time"):
    if target in da.dims:
        return da
    time_like = [d for d in da.dims if d.lower() == "time"]
    if len(time_like) == 1:
        return da.rename({time_like[0]: target})
    return da


def ensure_time_lat_lon_shape(da, lat_val=np.nan, lon_val=np.nan):
    """Force DataArray to [time, lat, lon] (singleton lat/lon allowed)."""
    da = squeeze_singleton_dims(da)
    da = ensure_time_dim_name(da, target="time")

    if "time" not in da.dims:
        raise ValueError(f"Expected time dimension, got dims={da.dims}")

    if "lat" not in da.dims:
        da = da.expand_dims(lat=[lat_val])
    if "lon" not in da.dims:
        da = da.expand_dims(lon=[lon_val])

    da = da.transpose("time", "lat", "lon")

    if da.sizes["lat"] == 1:
        da = da.assign_coords(lat=[lat_val])
    if da.sizes["lon"] == 1:
        da = da.assign_coords(lon=[lon_val])

    return da


def const_3d_time_lat_lon(time_coord, lat_val, lon_val, value, dtype=np.float32):
    data = np.full((time_coord.size, 1, 1), value, dtype=dtype)
    return xr.DataArray(
        data,
        dims=("time", "lat", "lon"),
        coords={"time": time_coord, "lat": [lat_val], "lon": [lon_val]},
    )


def const_3d_depth_lat_lon(depth_vals, lat_val, lon_val, value, dtype=np.float32):
    data = np.full((len(depth_vals), 1, 1), value, dtype=dtype)
    return xr.DataArray(
        data,
        dims=("depth", "lat", "lon"),
        coords={"depth": depth_vals, "lat": [lat_val], "lon": [lon_val]},
    )


def ensure_time_lat_lon_broadcast(da, time_coord, lat_val=np.nan, lon_val=np.nan):
    """
    Ensure DataArray has dims [time, lat, lon]:
      - [lat,lon] -> broadcast over time
      - [time,lat,lon] -> reorder
      - scalar -> constant [time,lat,lon]
    """
    da = squeeze_singleton_dims(da)
    da = ensure_time_dim_name(da, target="time")

    if da.ndim == 0:
        return const_3d_time_lat_lon(time_coord, lat_val, lon_val, da.values.item(), dtype=da.dtype)

    dims_set = set(da.dims)

    if dims_set == {"lat", "lon"}:
        da = da.expand_dims(time=time_coord).transpose("time", "lat", "lon")
        if da.sizes["lat"] == 1:
            da = da.assign_coords(lat=[lat_val])
        if da.sizes["lon"] == 1:
            da = da.assign_coords(lon=[lon_val])
        return da

    if dims_set == {"time", "lat", "lon"}:
        da = da.transpose("time", "lat", "lon")
        if da.sizes["lat"] == 1:
            da = da.assign_coords(lat=[lat_val])
        if da.sizes["lon"] == 1:
            da = da.assign_coords(lon=[lon_val])
        return da

    raise ValueError(f"Unexpected dims for time broadcast: {da.dims}")


def pick_single_pixel_and_standardize_xy_to_latlon(ds_in, site_x=None, site_y=None, lat_val=np.nan, lon_val=np.nan):
    """
    For datasets with dims like [time, y, x], [time, x, y], or [time] only:
      - pick one pixel (first or nearest x/y if site_x/site_y given)
      - rename x->lon, y->lat
      - force vars with time to [time, lat, lon]
    Returns a new dataset.
    """
    ds = ds_in.copy()

    # squeeze singleton dims everywhere
    for v in list(ds.data_vars):
        ds[v] = squeeze_singleton_dims(ds[v])

    # normalize time coord name if needed
    if "time" not in ds.coords:
        time_like_coords = [c for c in ds.coords if c.lower() == "time"]
        if len(time_like_coords) == 1:
            ds = ds.rename({time_like_coords[0]: "time"})

    # If x/y coords exist, choose one pixel
    has_x = "x" in ds.coords
    has_y = "y" in ds.coords

    if has_x and has_y:
        if (site_x is not None) and (site_y is not None):
            ds = ds.sel(x=site_x, y=site_y, method="nearest")
        else:
            # first pixel
            if ds["x"].size > 1 or ds["y"].size > 1:
                ds = ds.isel(x=0, y=0)

        # rename coords to lat/lon semantic dims
        rename_map = {}
        if "y" in ds.dims:
            rename_map["y"] = "lat"
        if "x" in ds.dims:
            rename_map["x"] = "lon"
        if rename_map:
            ds = ds.rename(rename_map)

    # force all time-varying vars to [time,lat,lon]
    for v in list(ds.data_vars):
        da = ds[v]
        da = ensure_time_dim_name(da, "time")
        if "time" in da.dims:
            # If var still has lat/lon >1, pick first
            indexers = {}
            if "lat" in da.dims and da.sizes["lat"] > 1:
                indexers["lat"] = 0
            if "lon" in da.dims and da.sizes["lon"] > 1:
                indexers["lon"] = 0
            if indexers:
                da = da.isel(indexers)

            da = ensure_time_lat_lon_shape(da, lat_val=lat_val, lon_val=lon_val)
            ds[v] = da

    return ds


def merge_extra_zarr_timeseries(
    ds_base,
    zarr_path,
    prefix,
    lat_val,
    lon_val,
    variable_whitelist=None,
    site_x=None,
    site_y=None,
    time_join="outer",
):
    """
    Merge vars from an extra zarr (e.g., Sentinel-2) into ds_base.
    - Standardizes extra vars to [time,lat,lon] by selecting a single pixel
    - Prefixes variable names to avoid collisions
    - Aligns times with base dataset using xr.align(join=time_join)
    """
    try:
        ds_extra = xr.open_zarr(zarr_path, consolidated=False)
    except Exception:
        ds_extra = xr.open_zarr(zarr_path, consolidated=True)

    ds_extra = pick_single_pixel_and_standardize_xy_to_latlon(
        ds_extra, site_x=site_x, site_y=site_y, lat_val=lat_val, lon_val=lon_val
    )

    # Keep selected vars only
    vars_to_merge = list(ds_extra.data_vars)
    if variable_whitelist is not None:
        vars_to_merge = [v for v in vars_to_merge if v in variable_whitelist]

    if len(vars_to_merge) == 0:
        print(f"[{prefix}] No variables selected to merge from {zarr_path}")
        return ds_base

    # Build renamed dataset with prefixed vars
    rename_map = {v: f"{prefix}_{v}" for v in vars_to_merge}
    ds_extra_sel = ds_extra[vars_to_merge].rename(rename_map)

    # Preserve attributes and add provenance tags
    for old_v, new_v in rename_map.items():
        ds_extra_sel[new_v].attrs.update(ds_extra[old_v].attrs)
        ds_extra_sel[new_v].attrs["source_zarr"] = zarr_path
        ds_extra_sel[new_v].attrs["source_variable"] = old_v
        ds_extra_sel[new_v].attrs["spatial_reduction"] = "single pixel (first or nearest x,y)"

    # Align with base time axis (important if timestamps differ)
    ds_base_aligned, ds_extra_aligned = xr.align(ds_base, ds_extra_sel, join=time_join)

    # Merge
    ds_merged = xr.merge([ds_base_aligned, ds_extra_aligned], compat="override", join="outer")
    print(f"[{prefix}] Merged variables: {list(rename_map.values())}")
    return ds_merged


def merge_extra_zarr_spatial_mean_timeseries(
    ds_base,
    zarr_path,
    prefix,
    lat_val,
    lon_val,
    variable_whitelist=None,
    time_join="outer",
    spatial_dims=("x", "y"),
    skipna=True,
):
    """
    Merge vars from an extra zarr after spatially averaging over x/y at each time step.
    Output vars are standardized to [time,lat,lon] singleton grid.
    Intended for biomass-like products.
    """
    try:
        ds_extra = xr.open_zarr(zarr_path, consolidated=False)
    except Exception:
        ds_extra = xr.open_zarr(zarr_path, consolidated=True)

    # squeeze singleton dims
    for v in list(ds_extra.data_vars):
        ds_extra[v] = squeeze_singleton_dims(ds_extra[v])

    # normalize time coord name
    if "time" not in ds_extra.coords:
        time_like_coords = [c for c in ds_extra.coords if c.lower() == "time"]
        if len(time_like_coords) == 1:
            ds_extra = ds_extra.rename({time_like_coords[0]: "time"})

    # choose vars
    vars_to_merge = list(ds_extra.data_vars)
    if variable_whitelist is not None:
        vars_to_merge = [v for v in vars_to_merge if v in variable_whitelist]

    if len(vars_to_merge) == 0:
        print(f"[{prefix}] No variables selected to merge from {zarr_path}")
        return ds_base

    out_vars = {}
    for v in vars_to_merge:
        da = ds_extra[v]
        da = squeeze_singleton_dims(da)
        da = ensure_time_dim_name(da, "time")

        if "time" not in da.dims:
            print(f"[{prefix}] Skip {v}: no time dimension (dims={da.dims})")
            continue

        # Average over x/y if present
        dims_to_mean = [d for d in spatial_dims if d in da.dims]
        if len(dims_to_mean) > 0:
            da_mean = da.mean(dim=dims_to_mean, skipna=skipna)
        else:
            da_mean = da

        # If any other non-time dims remain, reduce them too (defensive)
        residual_non_time_dims = [d for d in da_mean.dims if d != "time"]
        if len(residual_non_time_dims) > 0:
            da_mean = da_mean.mean(dim=residual_non_time_dims, skipna=skipna)

        # Convert [time] -> [time,lat,lon]
        da_out = ensure_time_lat_lon_shape(da_mean, lat_val=lat_val, lon_val=lon_val)

        new_name = f"{prefix}_{v}"
        out_vars[new_name] = da_out
        out_vars[new_name].attrs.update(da.attrs)
        out_vars[new_name].attrs["source_zarr"] = zarr_path
        out_vars[new_name].attrs["source_variable"] = v
        out_vars[new_name].attrs["spatial_reduction"] = "mean over x,y per time step"

    if len(out_vars) == 0:
        print(f"[{prefix}] No usable time-varying variables after spatial mean.")
        return ds_base

    ds_extra_mean = xr.Dataset(out_vars)

    ds_base_aligned, ds_extra_aligned = xr.align(ds_base, ds_extra_mean, join=time_join)
    ds_merged = xr.merge([ds_base_aligned, ds_extra_aligned], compat="override", join="outer")
    print(f"[{prefix}] Merged spatial-mean variables: {list(ds_extra_mean.data_vars)}")
    return ds_merged


def merge_s2_ndvi_spatial_mean(
    ds_base,
    sen2_zarr_path,
    lat_val,
    lon_val,
    out_name="s2_NDVI",
    time_join="outer",
    spatial_dims=("x", "y"),
    skipna=True,
    apply_scl_mask=False,
):
    """
    Compute Sentinel-2 NDVI = (B08 - B04)/(B08 + B04) on the ORIGINAL Sentinel cube,
    then spatially average over x,y for each time step, and merge into ds_base as
    [time, lat, lon] singleton grid.

    Notes
    -----
    - Uses B08 (NIR) and B04 (Red)
    - If apply_scl_mask=True and SCL exists, masks common cloud/shadow/water/snow classes
      before spatial averaging.
    """
    try:
        ds_s2 = xr.open_zarr(sen2_zarr_path, consolidated=False)
    except Exception:
        ds_s2 = xr.open_zarr(sen2_zarr_path, consolidated=True)

    # squeeze singleton dims
    for v in list(ds_s2.data_vars):
        ds_s2[v] = squeeze_singleton_dims(ds_s2[v])

    # normalize time coord name
    if "time" not in ds_s2.coords:
        time_like_coords = [c for c in ds_s2.coords if c.lower() == "time"]
        if len(time_like_coords) == 1:
            ds_s2 = ds_s2.rename({time_like_coords[0]: "time"})

    if ("B08" not in ds_s2.data_vars) or ("B04" not in ds_s2.data_vars):
        print("[s2_ndvi] Skip: B08 and/or B04 not found in Sentinel dataset")
        return ds_base

    nir = ensure_time_dim_name(squeeze_singleton_dims(ds_s2["B08"]), "time").astype(np.float32)
    red = ensure_time_dim_name(squeeze_singleton_dims(ds_s2["B04"]), "time").astype(np.float32)

    if "time" not in nir.dims or "time" not in red.dims:
        print(f"[s2_ndvi] Skip: B08/B04 missing time dim (B08 dims={nir.dims}, B04 dims={red.dims})")
        return ds_base

    # Compute NDVI per pixel, per time
    denom = nir + red
    ndvi = xr.where(denom != 0, (nir - red) / denom, np.nan).astype(np.float32)
    ndvi = ndvi.clip(min=-1.0, max=1.0)

    # Optional SCL mask before averaging
    if apply_scl_mask and ("SCL" in ds_s2.data_vars):
        scl = ensure_time_dim_name(squeeze_singleton_dims(ds_s2["SCL"]), "time").astype(np.int16)
        bad_scl = (
            (scl == 3) |   # cloud shadow
            (scl == 6) |   # water
            (scl == 8) |   # cloud medium probability
            (scl == 9) |   # cloud high probability
            (scl == 10) |  # cirrus
            (scl == 11)    # snow/ice
        )
        ndvi = ndvi.where(~bad_scl)

    # Spatial mean over x,y if present
    dims_to_mean = [d for d in spatial_dims if d in ndvi.dims]
    if len(dims_to_mean) > 0:
        ndvi_mean = ndvi.mean(dim=dims_to_mean, skipna=skipna)
    else:
        ndvi_mean = ndvi

    # Defensive reduction of any leftover non-time dims
    residual_non_time_dims = [d for d in ndvi_mean.dims if d != "time"]
    if len(residual_non_time_dims) > 0:
        ndvi_mean = ndvi_mean.mean(dim=residual_non_time_dims, skipna=skipna)

    # [time] -> [time,lat,lon]
    ndvi_out = ensure_time_lat_lon_shape(ndvi_mean, lat_val=lat_val, lon_val=lon_val).astype(np.float32)

    ndvi_out.attrs.update({
        "standard_name": "normalized_difference_vegetation_index",
        "long_name": f"Sentinel-2 NDVI ({'SCL-masked, ' if apply_scl_mask and ('SCL' in ds_s2.data_vars) else ''}spatial mean over x,y)",
        "units": "1",
        "source_zarr": sen2_zarr_path,
        "source_variable": "B08,B04" + (",SCL" if apply_scl_mask and ("SCL" in ds_s2.data_vars) else ""),
        "formula": "(B08 - B04) / (B08 + B04)",
        "nir_band": "B08",
        "red_band": "B04",
        "spatial_reduction": "mean over x,y per time step",
        "note": "NDVI computed per pixel first, then spatially averaged",
    })
    if apply_scl_mask and ("SCL" in ds_s2.data_vars):
        ndvi_out.attrs["masking"] = "Masked SCL classes [3,6,8,9,10,11] before spatial mean"

    ds_tmp = xr.Dataset({out_name: ndvi_out})

    ds_base_aligned, ds_tmp_aligned = xr.align(ds_base, ds_tmp, join=time_join)
    ds_merged = xr.merge([ds_base_aligned, ds_tmp_aligned], compat="override", join="outer")

    print(f"[s2_ndvi] Merged {out_name} (spatial mean NDVI over x,y)")
    return ds_merged


# -----------------------------------------------------------------------------
# 1) Open ERA5-Land dataset
# -----------------------------------------------------------------------------
ds = xr.open_zarr(era5_zarr_path, consolidated=False)

# Squeeze singleton dims in ERA5 vars
for v in list(ds.data_vars):
    ds[v] = squeeze_singleton_dims(ds[v])

# Ensure time coordinate exists
if "time" not in ds.coords:
    time_like_coords = [c for c in ds.coords if c.lower() == "time"]
    if len(time_like_coords) == 1:
        ds = ds.rename({time_like_coords[0]: "time"})
    else:
        raise ValueError("No 'time' coordinate found in ERA5 dataset.")

time_coord = ds["time"]


# -----------------------------------------------------------------------------
# 2) Select single ERA5 grid cell and force source vars to [time,lat,lon]
# -----------------------------------------------------------------------------
if ("lat" in ds.coords) and ("lon" in ds.coords):
    if (site_lat is not None) and (site_lon is not None):
        ds = ds.sel(lat=site_lat, lon=site_lon, method="nearest")
        lat_val = float(ds["lat"].values) if np.ndim(ds["lat"].values) == 0 else float(ds["lat"].values[0])
        lon_val = float(ds["lon"].values) if np.ndim(ds["lon"].values) == 0 else float(ds["lon"].values[0])
    else:
        if ds["lat"].size > 1 or ds["lon"].size > 1:
            ds = ds.isel(lat=0, lon=0)
        lat_val = float(ds["lat"].values) if np.ndim(ds["lat"].values) == 0 else float(ds["lat"].values[0])
        lon_val = float(ds["lon"].values) if np.ndim(ds["lon"].values) == 0 else float(ds["lon"].values[0])
else:
    lat_val, lon_val = np.nan, np.nan

# Re-squeeze and standardize ERA5 time-varying vars
for v in list(ds.data_vars):
    da = ds[v]
    da = squeeze_singleton_dims(da)
    da = ensure_time_dim_name(da, "time")
    if "time" in da.dims:
        ds[v] = ensure_time_lat_lon_shape(da, lat_val=lat_val, lon_val=lon_val)
    else:
        ds[v] = da

# Refresh time coord after ERA5 processing
time_coord = ds["time"]


# -----------------------------------------------------------------------------
# 3) Compute derived ERA5 forcing variables
# -----------------------------------------------------------------------------
t2m = ds["t2m"] if "t2m" in ds else None
d2m = ds["d2m"] if "d2m" in ds else None
tp = ds["tp"] if "tp" in ds else None
ssrd = ds["ssrd"] if "ssrd" in ds else None
ssr = ds["ssr"] if "ssr" in ds else None          # net shortwave (J m-2 timestep-1)
str_ = ds["str"] if "str" in ds else None         # net longwave  (J m-2 timestep-1)

# f_airT [°C]
if t2m is not None:
    ds = add_if_missing(ds, "f_airT", (t2m - 273.15).astype(np.float32), attrs={
        "standard_name": "Tair",
        "units": "°C",
        "source_variable": "t2m",
        "note": "Converted from K to °C",
    })

# f_airT_day [°C] fallback
if "f_airT" in ds:
    ds = add_if_missing(ds, "f_airT_day", ds["f_airT"].astype(np.float32), attrs={
        "standard_name": "TairDay",
        "units": "°C",
        "source_variable": "t2m",
        "note": "Fallback = f_airT (no separate daytime T available)",
    })

# f_VPD [kPa], f_VPD_day [kPa] (Magnus formula)
if (t2m is not None) and (d2m is not None):
    t_c = t2m - 273.15
    td_c = d2m - 273.15

    es = 611.2 * np.exp((17.67 * t_c) / (t_c + 243.5))      # Pa
    ea = 611.2 * np.exp((17.67 * td_c) / (td_c + 243.5))    # Pa

    vpd_pa = xr.where((es - ea) < 0, 0, es - ea)
    vpd_kpa = (vpd_pa / 1000.0).astype(np.float32)

    ds = add_if_missing(ds, "f_VPD", vpd_kpa, attrs={
        "standard_name": "Vapor pressure deficit",
        "units": "kPa",
        "source_variable": "t2m,d2m",
        "formula": "VPD = es(Tair)-ea(Tdew), Magnus equation",
    })
    ds = add_if_missing(ds, "f_VPD_day", vpd_kpa, attrs={
        "standard_name": "Vapor pressure deficit Day",
        "units": "kPa",
        "source_variable": "t2m,d2m",
        "note": "Fallback = f_VPD",
    })

# f_rg [MJ m-2 timestep-1], f_PAR [MJ m-2 timestep-1]
if ssrd is not None:
    f_rg = (ssrd / 1e6).astype(np.float32)
    ds = add_if_missing(ds, "f_rg", f_rg, attrs={
        "standard_name": "Global Radiation",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Converted from J m-2 to MJ m-2; timestep accumulation",
    })
    ds = add_if_missing(ds, "f_PAR", (0.5 * f_rg).astype(np.float32), attrs={
        "standard_name": "Photosynthetically active radiation (=Rg*.5)",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Approximated as 0.5 * f_rg",
    })

# f_rn [MJ m-2 timestep-1] (net radiation)
if (ssr is not None) and (str_ is not None):
    f_rn = ((ssr + str_) / 1e6).astype(np.float32)
    ds = add_if_missing(ds, "f_rn", f_rn, attrs={
        "standard_name": "Net radiation",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssr,str",
        "formula": "Rn = (SSR + STR) / 1e6",
        "note": "ERA5 accumulated net shortwave + net longwave over timestep",
    })
elif "f_rg" in ds:
    ds = add_if_missing(ds, "f_rn", ds["f_rg"].astype(np.float32), attrs={
        "standard_name": "Net radiation (approximate)",
        "units": "MJ m-2 timestep-1",
        "source_variable": "ssrd",
        "note": "Fallback approximation f_rn = f_rg because ssr/str not available",
    })

# f_rain [mm timestep-1]
if tp is not None:
    ds = add_if_missing(ds, "f_rain", (tp * 1000.0).astype(np.float32), attrs={
        "standard_name": "Rain",
        "units": "mm timestep-1",
        "source_variable": "tp",
        "note": "Converted from m to mm; timestep accumulation",
    })


# -----------------------------------------------------------------------------
# 4) Add missing constants in required shapes
# -----------------------------------------------------------------------------
spatiotemporal_constants = {
    "f_ambient_CO2": (
        CONST["f_ambient_CO2"], np.float32,
        {"standard_name": "ambient_CO2", "units": "ppm", "source_variable": "constant_fill"}
    ),
}

surface_constants_time_broadcast = {
    "f_frac_vegetation": (
        CONST["f_frac_vegetation"], np.float32,
        {"standard_name": "vegetation fraction", "units": "-", "source_variable": "constant_fill",
         "note": "static variable broadcasted to [time,lat,lon]"}
    ),
    "f_tree_frac": (
        CONST["f_tree_frac"], np.float32,
        {"standard_name": "tree fraction", "units": "-", "source_variable": "constant_fill",
         "note": "static variable broadcasted to [time,lat,lon]"}
    ),
    "f_pft": (
        CONST["f_pft"], np.int16,
        {"standard_name": "pft index", "units": "-", "source_variable": "constant_fill",
         "pft_label": "SAV", "pft_index_mapping": "SindbadML.PFTlabels (1-based)",
         "note": "static categorical variable broadcasted to [time,lat,lon]"}
    ),
    "f_dist_intensity": (
        CONST["f_dist_intensity"], np.int16,
        {"standard_name": "isDisturbed flag for disturbance", "units": "-", "source_variable": "constant_fill",
         "note": "static categorical variable broadcasted to [time,lat,lon]"}
    ),
    "f_burnt_area": (
        CONST["f_burnt_area"], np.float32,
        {"standard_name": "fire fraction area per area pixel", "units": "-", "source_variable": "constant_fill",
         "note": "static variable broadcasted to [time,lat,lon]"}
    ),
}

soil_profile_constants = {
    "f_clay": (CONST["f_clay"], np.float32, {"standard_name": "CLAY", "units": "-", "source_variable": "constant_fill"}),
    "f_sand": (CONST["f_sand"], np.float32, {"standard_name": "SAND", "units": "-", "source_variable": "constant_fill"}),
    "f_silt": (CONST["f_silt"], np.float32, {"standard_name": "SILT", "units": "-", "source_variable": "constant_fill"}),
    "f_orgm": (CONST["f_orgm"], np.float32, {"standard_name": "organic matter", "units": "-", "source_variable": "constant_fill"}),
}

for name, (val, dtype, attrs) in spatiotemporal_constants.items():
    if name not in ds:
        ds = add_if_missing(ds, name, const_3d_time_lat_lon(time_coord, lat_val, lon_val, val, dtype), attrs=attrs)

for name, (val, dtype, attrs) in surface_constants_time_broadcast.items():
    if name not in ds:
        ds = add_if_missing(ds, name, const_3d_time_lat_lon(time_coord, lat_val, lon_val, val, dtype), attrs=attrs)

for name, (val, dtype, attrs) in soil_profile_constants.items():
    if name not in ds:
        ds = add_if_missing(ds, name, const_3d_depth_lat_lon(SOIL_DEPTH_CM, lat_val, lon_val, val, dtype), attrs=attrs)

for v in ["f_pft", "f_tree_frac", "f_frac_vegetation", "f_dist_intensity", "f_burnt_area"]:
    if v in ds and ds[v].dims != ("time", "lat", "lon"):
        ds[v] = ensure_time_lat_lon_broadcast(ds[v], time_coord, lat_val=lat_val, lon_val=lon_val)
        print(f"Broadcasted {v} -> [time,lat,lon]: {ds[v].shape}")


# -----------------------------------------------------------------------------
# 5) Merge biomass zarr (CCI biomass) into ds as spatial mean over x,y
# -----------------------------------------------------------------------------
try:
    ds = merge_extra_zarr_spatial_mean_timeseries(
        ds_base=ds,
        zarr_path=biomass_zarr_path,
        prefix="cci",
        lat_val=lat_val,
        lon_val=lon_val,
        variable_whitelist=["agb", "agb_sd"],
        time_join="outer",
        spatial_dims=("x", "y"),
        skipna=True,
    )
except Exception as e:
    print(f"WARNING: biomass spatial-mean merge failed: {e}")


# -----------------------------------------------------------------------------
# 6) Merge Sentinel-2 zarr into ds (single pixel selection for raw bands)
# -----------------------------------------------------------------------------
s2_vars = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B09", "B11", "B12", "B8A", "SCL"]

try:
    ds = merge_extra_zarr_timeseries(
        ds_base=ds,
        zarr_path=sen2_zarr_path,
        prefix="s2",
        lat_val=lat_val,
        lon_val=lon_val,
        variable_whitelist=s2_vars,
        site_x=sen2_site_x,
        site_y=sen2_site_y,
        time_join="outer",
    )
except Exception as e:
    print(f"WARNING: Sentinel-2 merge failed: {e}")


# -----------------------------------------------------------------------------
# 6b) Compute Sentinel-2 NDVI as SPATIAL MEAN over x,y (per time step)
# -----------------------------------------------------------------------------
# NDVI is computed on the ORIGINAL Sentinel cube (per pixel), THEN averaged spatially.
# This is preferable to NDVI from a single pixel.
# -----------------------------------------------------------------------------
try:
    ds = merge_s2_ndvi_spatial_mean(
        ds_base=ds,
        sen2_zarr_path=sen2_zarr_path,
        lat_val=lat_val,
        lon_val=lon_val,
        out_name="s2_NDVI",
        time_join="outer",
        spatial_dims=("x", "y"),
        skipna=True,
        apply_scl_mask=False,   # set True if you only want masked NDVI
    )
except Exception as e:
    print(f"WARNING: Sentinel-2 NDVI spatial-mean merge failed: {e}")

# Optional masked NDVI as an additional variable
try:
    ds = merge_s2_ndvi_spatial_mean(
        ds_base=ds,
        sen2_zarr_path=sen2_zarr_path,
        lat_val=lat_val,
        lon_val=lon_val,
        out_name="s2_NDVI_masked",
        time_join="outer",
        spatial_dims=("x", "y"),
        skipna=True,
        apply_scl_mask=True,
    )
except Exception as e:
    print(f"WARNING: Sentinel-2 NDVI (SCL-masked) spatial-mean merge failed: {e}")


# -----------------------------------------------------------------------------
# 7) Optional aliases
# -----------------------------------------------------------------------------
if ("f_ambient_CO2" in ds) and ("f_CO2" not in ds):
    ds["f_CO2"] = ds["f_ambient_CO2"]
    ds["f_CO2"].attrs.update({"units": "ppm", "note": "Alias of f_ambient_CO2"})
    print("Added alias: f_CO2 -> f_ambient_CO2")

if ("f_rg" in ds) and ("SW_IN_ERAIv2_gfld" not in ds):
    ds["SW_IN_ERAIv2_gfld"] = ds["f_rg"]
    ds["SW_IN_ERAIv2_gfld"].attrs.update({"note": "Alias of f_rg"})
    print("Added alias: SW_IN_ERAIv2_gfld -> f_rg")

if ("f_rn" in ds) and ("RNET_ERAIv2_gfld" not in ds):
    ds["RNET_ERAIv2_gfld"] = ds["f_rn"]
    ds["RNET_ERAIv2_gfld"].attrs.update({"note": "Alias of f_rn"})
    print("Added alias: RNET_ERAIv2_gfld -> f_rn")

if ("f_rain" in ds) and ("P_ERAIv2_gfld" not in ds):
    ds["P_ERAIv2_gfld"] = ds["f_rain"]
    ds["P_ERAIv2_gfld"].attrs.update({"note": "Alias of f_rain"})
    print("Added alias: P_ERAIv2_gfld -> f_rain")


# -----------------------------------------------------------------------------
# 8) Final checks
# -----------------------------------------------------------------------------
print("\n================ FINAL DATASET SUMMARY ================")
print(ds)

print("\n================ VARIABLE SHAPES ======================")
for v in ds.data_vars:
    print(f"{v:20s} dims={ds[v].dims} shape={ds[v].shape} dtype={ds[v].dtype} units={ds[v].attrs.get('units', 'NA')}")

print("\n=========== CORE FORCING SHAPE CHECKS ================")
for v in [
    "f_airT", "f_airT_day", "f_VPD", "f_VPD_day", "f_rain", "f_rg", "f_rn", "f_PAR", "f_ambient_CO2",
    "f_pft", "f_tree_frac", "f_frac_vegetation", "f_dist_intensity", "f_burnt_area"
]:
    if v in ds:
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims}")

for v in ["f_clay", "f_sand", "f_silt", "f_orgm"]:
    if v in ds:
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims}, depth={ds[v]['depth'].values}")

print("\n=========== REMOTE SENSING MERGED VARS ===============")
for v in ds.data_vars:
    if v.startswith("cci_") or v.startswith("s2_"):
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims} | spatial_reduction={ds[v].attrs.get('spatial_reduction', 'NA')}")

for v in ["cci_agb", "cci_agb_sd"]:
    if v in ds:
        print(f"{v} attrs: {ds[v].attrs}")

for v in ["s2_NDVI", "s2_NDVI_masked"]:
    if v in ds:
        vmin = ds[v].min(skipna=True).values
        vmax = ds[v].max(skipna=True).values
        print(f"{v:18s} -> {ds[v].shape} {ds[v].dims} min={float(vmin):.3f} max={float(vmax):.3f}")
        print(f"{v} attrs: {ds[v].attrs}")

# -----------------------------------------------------------------------------
# 9) Optional save
# -----------------------------------------------------------------------------
# out_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_forcing_plus_rs.zarr"
# ds.to_zarr(out_path, mode='w')
# print(f"Saved to: {out_path}")

Added: f_airT               dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_airT_day           dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_VPD                dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_VPD_day            dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_rg                 dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_PAR                dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_rn                 dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_rain               dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_ambient_CO2        dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_frac_vegetation    dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_tree_frac          dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_pft                dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_dist_intensity     dims=('time', 'lat', 'lon') shape=(8784, 1, 1)
Added: f_burnt_area      

## 7. Finalize, rechunk, and write the merged dataset

This section prepares the final dataset for storage:
- copies the dataset
- drops non-essential metadata variables
- rechunks for efficient Zarr writing / downstream access
- writes the result to the configured output path

After writing, check the printed path to confirm the file was saved where expected.


In [6]:
ds_out = ds.copy()

# Drop metadata vars that are not needed for forcing
drop_vars = [v for v in ["expver", "number", "spatial_ref"] if v in ds_out.variables]
ds_out = ds_out.drop_vars(drop_vars, errors="ignore")

# Rechunk after dropping
ds_out = ds_out.chunk({"time": 365, "lat": 1, "lon": 1})

out_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land_sindbad_forcing_obs.zarr"
ds_out.to_zarr(out_path, mode="w", align_chunks=True)
print(f"Saved to: {out_path}")

/tmp/ipykernel_3795681/605834019.py:11: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  ds_out.to_zarr(out_path, mode="w", align_chunks=True)
/tmp/ipykernel_3795681/605834019.py:11: UserWarning: Times can't be serialized faithfully to int64 with requested units 'seconds since 1970-01-01'. Serializing with units 'milliseconds since 1970-01-01' instead. Set encoding['dtype'] to floating point dtype to serialize with units 'seconds since 1970-01-01'. Set encoding['units'] to 'milliseconds since 1970-01-01' to silence this warning .
  ds_out.to_zarr(out_path, mode="w", align_chunks=True)


Saved to: /Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land_sindbad_forcing_obs.zarr


## 8. Aggregate the data to daily scale



In [9]:
# -----------------------------------------------------------------------------
# 10) Convert hourly zarr -> daily zarr
#     - Fluxes: hourly values accumulated to daily sums
#     - States: hourly values averaged to daily means
# -----------------------------------------------------------------------------
import os
import xarray as xr
import numpy as np

# Input can be the in-memory ds_out OR reopen from disk
# hourly_in_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land_sindbad_forcing_obs.zarr"
# ds_hourly = xr.open_zarr(hourly_in_path, consolidated=False)
ds_hourly = ds_out.copy()

# ---- User-defined classification ----
# Variables treated as FLUX/ACCUMULATION over time (sum to daily)
# (Add/remove names as needed for your workflow)
flux_vars = {
    # ERA5/SINDBAD forcing flux-like variables created above
    "f_rain",                 # mm timestep-1 -> mm day-1 (sum)
    "f_rg",                   # MJ m-2 timestep-1 -> MJ m-2 day-1 (sum)
    "f_PAR",                  # MJ m-2 timestep-1 -> MJ m-2 day-1 (sum)
    "f_rn",                   # MJ m-2 timestep-1 -> MJ m-2 day-1 (sum)
    "ssr",
    "ssrd",
    "tp",
    # aliases (if present)
    "SW_IN_ERAIv2_gfld",
    "RNET_ERAIv2_gfld",
    "P_ERAIv2_gfld",

    # add other flux variables here if they exist in the dataset
    # e.g., "GPP", "RECO", "ET", ...
}

# Optional explicit state vars (not required; default behavior is mean for time-varying non-flux vars)
# state_vars = {...}

# ---- Helper functions ----
def _normalize_time_coord(ds):
    """Ensure time coord exists and is datetime64."""
    if "time" not in ds.coords:
        raise ValueError("Dataset has no 'time' coordinate.")
    if not np.issubdtype(ds["time"].dtype, np.datetime64):
        ds = ds.assign_coords(time=xr.decode_cf(ds).time)
    return ds

def _daily_reduce_dataarray(da, how="mean"):
    """
    Daily reduce a DataArray with time dimension.
    - how='sum': for fluxes/accumulations
    - how='mean': for states
    Keeps dimensions other than time intact.
    """
    if "time" not in da.dims:
        return da

    # Use resample on day boundary
    if how == "sum":
        out = da.resample(time="1D").sum(skipna=True)
    elif how == "mean":
        out = da.resample(time="1D").mean(skipna=True)
    else:
        raise ValueError(f"Unknown reduction method: {how}")

    # preserve attrs
    out.attrs = da.attrs.copy()

    # Update metadata note
    old_note = out.attrs.get("note", "")
    if how == "sum":
        extra_note = "Daily sum aggregated from hourly values."
        # Optional unit tweak for timestep/day wording
        units = out.attrs.get("units", "")
        if isinstance(units, str):
            units = units.replace("timestep-1", "day-1")
            out.attrs["units"] = units
    else:
        extra_note = "Daily mean aggregated from hourly values."

    out.attrs["temporal_aggregation"] = "daily"
    out.attrs["aggregation_method"] = how
    out.attrs["note"] = f"{old_note} | {extra_note}".strip(" |")

    return out

def make_daily_dataset(ds_hourly, flux_vars):
    ds_hourly = _normalize_time_coord(ds_hourly)

    out_vars = {}
    for v in ds_hourly.data_vars:
        da = ds_hourly[v]

        # No time dimension -> keep unchanged (static vars, soil profiles, etc.)
        if "time" not in da.dims:
            out_vars[v] = da
            continue

        # Decide reduction
        if v in flux_vars:
            out_vars[v] = _daily_reduce_dataarray(da, how="sum")
            print(f"[SUM ] {v}")
        else:
            out_vars[v] = _daily_reduce_dataarray(da, how="mean")
            print(f"[MEAN] {v}")

    ds_daily = xr.Dataset(out_vars, attrs=ds_hourly.attrs.copy())

    # Carry over non-dim coords safely (if they still align)
    for cname, c in ds_hourly.coords.items():
        if cname in ds_daily.coords:
            continue
        if "time" not in c.dims:
            ds_daily = ds_daily.assign_coords({cname: c})

    # Global attrs
    ds_daily.attrs["temporal_resolution"] = "daily"
    ds_daily.attrs["source_temporal_resolution"] = "hourly"
    ds_daily.attrs["daily_aggregation_rule"] = (
        "Flux variables summed; state variables averaged; static vars unchanged."
    )

    return ds_daily

# ---- Build daily dataset ----
ds_daily = make_daily_dataset(ds_hourly, flux_vars=flux_vars)

# Optional: drop metadata vars if somehow present
drop_vars_daily = [v for v in ["expver", "number", "spatial_ref"] if v in ds_daily.variables]
ds_daily = ds_daily.drop_vars(drop_vars_daily, errors="ignore")

# Rechunk for daily data (smaller time axis; adjust as desired)
if {"time", "lat", "lon"}.issubset(set(ds_daily.dims)):
    ds_daily = ds_daily.chunk({"time": 365, "lat": 1, "lon": 1})

# ---- Save with a new name ----
hourly_out_path = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land_sindbad_forcing_obs.zarr"
daily_out_path = hourly_out_path.replace(".zarr", "_daily.zarr")

ds_daily.to_zarr(daily_out_path, mode="w", align_chunks=True)
print(f"Saved daily zarr to: {daily_out_path}")

# ---- Quick checks ----
print("\n================ DAILY DATASET SUMMARY ================")
print(ds_daily)

print("\n=========== DAILY VARIABLE SHAPES / AGG METHOD ========")
for v in ds_daily.data_vars:
    agg = ds_daily[v].attrs.get("aggregation_method", "unchanged/no-time")
    units = ds_daily[v].attrs.get("units", "NA")
    print(f"{v:20s} dims={ds_daily[v].dims} shape={ds_daily[v].shape} agg={agg} units={units}")

[MEAN] d2m
[SUM ] ssr
[SUM ] ssrd
[MEAN] t2m
[SUM ] tp
[MEAN] f_airT
[MEAN] f_airT_day
[MEAN] f_VPD
[MEAN] f_VPD_day
[SUM ] f_rg
[SUM ] f_PAR
[SUM ] f_rn
[SUM ] f_rain
[MEAN] f_ambient_CO2
[MEAN] f_frac_vegetation
[MEAN] f_tree_frac
[MEAN] f_pft
[MEAN] f_dist_intensity
[MEAN] f_burnt_area
[MEAN] cci_agb
[MEAN] cci_agb_sd
[MEAN] s2_B01
[MEAN] s2_B02
[MEAN] s2_B03
[MEAN] s2_B04
[MEAN] s2_B05
[MEAN] s2_B06
[MEAN] s2_B07
[MEAN] s2_B08
[MEAN] s2_B09
[MEAN] s2_B11
[MEAN] s2_B12
[MEAN] s2_B8A
[MEAN] s2_SCL
[MEAN] s2_NDVI
[MEAN] s2_NDVI_masked
[MEAN] f_CO2
[SUM ] SW_IN_ERAIv2_gfld
[SUM ] RNET_ERAIv2_gfld
[SUM ] P_ERAIv2_gfld
Saved daily zarr to: /Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_era5land_sindbad_forcing_obs_daily.zarr

================ DAILY DATASET SUMMARY ================
<xarray.Dataset> Size: 63kB
Dimensions:            (lat: 1, lon: 1, time: 366, depth: 7)
Coordinates:
  * lat                (lat) float64 8B -15.26
  *

## 9. Re-open external biomass Zarr for inspection

This section opens the biomass-related Zarr store (`ccibiomass` in the current workflow) separately for direct inspection and validation.

Use this to verify:
- variable names (e.g., `agb`)
- dimensions / coordinates
- data coverage before or after merge


In [6]:
zarr_path_eebiomass = "/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Feb12/cube_generation/data/AU-Dry_ccibiomass.zarr"

# Open dataset
ds_eebiomass = xr.open_zarr(zarr_path_eebiomass, consolidated=False)  # try consolidated=True if needed


## 10. Quick dataframe view of biomass variable

Convert the `agb` variable to a pandas DataFrame for a compact tabular preview.  
This is useful for checking values, timestamps, and whether the variable is populated as expected.


In [38]:
ds_eebiomass.agb.to_dataframe()

spatial_ref       agb
time                y         x                              
2020-07-01 23:59:59 8312235.0 860115.0            0  7.432228
                              860125.0            0  7.174846
                              860135.0            0  6.916811
                              860145.0            0  6.658122
                              860155.0            0  6.398778
...                                             ...       ...
                    8308245.0 864065.0            0  0.000000
                              864075.0            0  0.000000
                              864085.0            0  0.000000
                              864095.0            0  0.000000
                              864105.0            0  0.000000

[160000 rows x 2 columns]

## 11. Sanity check non-NaN values in merged biomass field

This cell extracts the merged `cci_agb` array from `ds_out`, removes `NaN` values, and prints the remaining values.

This is a simple validation step to confirm that the merge produced actual numeric content (not all missing values).


In [31]:
arr = ds_out.cci_agb.squeeze(drop=True).values
print(arr[~np.isnan(arr)])
# print(ds_out.cci_agb.squeeze(drop=True).values)

[1.897344]


## 12. Notes / next steps

Suggested follow-up checks before using the output in Sindbad:
- verify **units** and **dimension names** for all required forcing variables
- confirm **time alignment** and temporal coverage
- inspect a few key variables (`t2m`, `tp`, radiation, CO2, `cci_agb`, Sentinel-2 fields)
- test-read the saved Zarr and confirm chunking/performance
- document the final output path in the experiment config (`experiment_json`)


## 13. Batch execute this notebook for all complete three-data-stream sites
This section is intended for **batch execution** of this notebook across multiple sites that have complete three-data-stream data (ERA5-Land, Sentinel-2, biomass). For now a list of 40 sites is hard-coded, but this can be replaced with a dynamic discovery of available sites in the future. Note that this section is **not intended for interactive use**; it is a template for HPC batch execution. And it is recommended to run this section in a separate notebook or script to avoid accidental execution during interactive sessions.

In [ ]:
# Set run_batch = True, then run this cell to execute one parameterized copy
# of this notebook per site. Child runs keep run_batch=False to avoid recursion.
from pathlib import Path
import sys

complete_site_ids = [
    "AR-SLu", "AR-Vir", "AT-Neu", "AU-ASM", "AU-Ade", "AU-Cum", "AU-DaP", "AU-DaS",
    "AU-Dry", "AU-Emr", "AU-Fog", "AU-Gin", "AU-RDF", "AU-Whr", "AU-Wom", "AU-Ync",
    "BE-Bra", "BE-Vie", "BR-Sa1", "CA-Gro", "CA-Man", "CA-NS2", "CA-NS5", "CA-NS6",
    "CA-Obs", "CA-Qfo", "CA-SF1", "CA-SF2", "CA-SF3", "CA-TP1", "CA-TP2", "CA-TP3",
]

if run_batch:
    try:
        import papermill as pm
    except ImportError as exc:
        raise ImportError(
            "Batch execution needs papermill in this Python environment. "
            "Install it or run this notebook with an environment that already has papermill."
        ) from exc

    notebook_path = Path("/Net/Groups/BGI/work_4/scratch/eebiomass/sindbad_eolincs/eo-lincs-scs3-Jul03/process_data_documented.ipynb")
    batch_notebook_dir = notebook_path.parent / "batch_runs"
    batch_notebook_dir.mkdir(parents=True, exist_ok=True)

    for batch_site_id in complete_site_ids:
        executed_notebook = batch_notebook_dir / f"process_data_documented_{batch_site_id}.ipynb"
        print(f"\n=== Running {batch_site_id} ===")
        pm.execute_notebook(
            input_path=str(notebook_path),
            output_path=str(executed_notebook),
            parameters={
                "site_id": batch_site_id,
                "data_root": str(data_root),
                "output_root": None if output_root is None else str(output_root),
                "run_batch": False,
                "site_lat": site_lat,
                "site_lon": site_lon,
                "sen2_site_x": sen2_site_x,
                "sen2_site_y": sen2_site_y,
            },
            kernel_name=None,
        )
        print(f"Saved executed notebook: {executed_notebook}")
else:
    print("Batch execution is disabled. Set run_batch = True before running this cell.")
